### Step 2 of the unintentional attack pipeline. Finding the preceding context, and prepare them for attacks

In [ ]:
### import all necessary libraries
import os
import pandas as pd
import numpy as np
import openai
import requests
import time
from tqdm import tqdm
from openai import OpenAI
from IPython.display import clear_output, display
from dotenv import dotenv_values
from dotenv import load_dotenv
tqdm.pandas()

In [ ]:
deepseek_api_key = os.environ["DEEPSEEK_API_KEY"]
client = OpenAI(api_key=deepseek_api_key, base_url="https://api.deepseek.com")

In [ ]:
prompt = """
The string given is a method snippet, in which there are two parts.
First part is the method signature, may include: method name, docstring, or comments.
Second part is the method body, the code itself.
Analyze the snippet, and return in this format: <first part>: extracted first part. <second part>: extracted second part.
DON'T leave anything out,keep all the space, new line, etc, don't add anything new, just return the result, do not say anything else.
Note that the second part ONLY contains the code, nothing else.
ALL parts are returned as they are, no changes, no modifications.
DO NOT generate anything new, just return the result, do not say anything else.
IF not about to find the two parts, return "Not found", don't generate anything else.
"""

In [ ]:
def separate_method_parts(client, prompt, string1):
    response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": f"{prompt}"},
        {"role": "user", "content": f"please analyze the method snippet: {string1}"},
    ],
    stream=False,
    temperature=0.0,
    )

    return response.choices[0].message.content

def part_separator(row):
    return separate_method_parts(
        client, 
        prompt, 
        row["all_methods"]
    )

In [ ]:
dfs = []
base_path = "/Users/anonymous_user/Downloads/experiment_results/exp4_attribute_analysis_results/step1_extracted_methods_raw_data/starcoder/7b"
for file in os.listdir(base_path):
    if file.endswith(".csv"):
        df = pd.read_csv(os.path.join(base_path, file))
        dfs.append(df)
df = pd.concat(dfs, ignore_index=True)


In [ ]:
df['separated_method_snippet'] = df.progress_apply(part_separator, axis=1)

In [ ]:
prompt1 = """ 
Follow the instructions below extremely strictly to complete the task:

We have two strings, string 1 is the content of a repo file, string 2 is a code snippet extracted from string 1.

Find the location of string 2 in string 1, and return 300 characteres of context precedding to it, no overlap, 300 chars before string 2 only. If there is less than 300 chars, return all of the preceding part.

DO NOT generate anything new, just find it in the string 1 verbatim, and return it directly, do not say anything else, just give the result.

If cannot find the string 2 in string 1, return "Not found", don't generate anything else.
"""

In [ ]:
def find_context(client, prompt, string1, string2):
    response = client.chat.completions.create(
    model="deepseek-chat",
    messages=[
        {"role": "system", "content": f"{prompt}"},
        {"role": "user", "content": f"string1 is {string1}, string2 is {string2}, analyze and return the result"},
    ],
    stream=False,
    temperature=0.0,
    )

    return response.choices[0].message.content

def context_finder(row):
    return find_context(
        client, 
        prompt1, 
        row["content"],
        row["all_methods"]
    )

In [ ]:
df['preceding_context'] = df.progress_apply(context_finder, axis=1)

In [ ]:
df.to_csv("/Users/anonymous_user/Downloads/experiment_results/exp4_attribute_analysis_results/step3_results_with_preceding_context_raw_data/sc_7b/step3_preceding_context_input.csv", index=False)